# Gate 2 Task 1 Preprocessing

This notebook prepares inputs for the GLM and XGBoost approaches. It learns rules from training claims and applies them unchanged to validation. It does not load loss values or evaluate model accuracy.

Run the cells from top to bottom. The checks cover rare and unseen labels, missing values, input columns, independent fitting, and prediction alignment. Model checks use artificial targets only.

The local Gate 1 plan is still marked proposed. These checks support team review; they are not Gate 2 approval.


## 1 Setup and recorded inputs

Use the project Python environment. If needed, install `scikit-learn==1.7.2` and `xgboost==3.1.1` with `%pip install` in a separate setup cell. On macOS, XGBoost also needs OpenMP (`brew install libomp` in Terminal).

The notebook finds the project from its current folder, so teammates do not need to edit a personal file path. It checks the saved split and file fingerprints before loading predictors. No packages are installed by running this notebook.


In [1]:
import hashlib
import json
import pickle
import sys
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.validation import check_is_fitted

ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data" / "allstate_claims_data.csv").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the project folder.")

SOURCE = ROOT / "data" / "allstate_claims_data.csv"
GATE1 = ROOT / "artifacts" / "october" / "gate 1"
MANIFEST = GATE1 / "october_partition_manifest.csv"
METADATA = GATE1 / "october_split_metadata.json"
RARE_THRESHOLD = 30

for path in [SOURCE, MANIFEST, METADATA]:
    assert path.is_file(), f"Required file missing: {path}"

print("Python:", sys.version.split()[0])
for package in ["pandas", "numpy", "scipy", "scikit-learn", "xgboost"]:
    print(f"{package}: {version(package)}")


Python: 3.13.2
pandas: 2.3.2
numpy: 2.3.3
scipy: 1.18.1
scikit-learn: 1.7.2
xgboost: 3.1.1


In [2]:
manifest = pd.read_csv(MANIFEST)

with METADATA.open() as file:
    metadata = json.load(file)

print("Manifest columns:", manifest.columns.tolist())
print("\nActual partition counts:")
print(manifest["partition"].value_counts())

print("\nExpected partition counts:")
print(metadata["partition_counts"])

print("\nDuplicate IDs:", manifest["id"].duplicated().sum())
print("Missing values:", manifest.isna().sum().to_dict())


Manifest columns: ['id', 'partition']

Actual partition counts:
partition
train         141238
validation     28248
test           18832
Name: count, dtype: int64

Expected partition counts:
{'test': 18832, 'train': 141238, 'validation': 28248}

Duplicate IDs: 0
Missing values: {'id': 0, 'partition': 0}


In [3]:
assert manifest.columns.tolist() == ["id", "partition"]
assert manifest["id"].is_unique
assert not manifest.isna().any().any()
assert manifest["partition"].value_counts().to_dict() == metadata["partition_counts"]
assert set(manifest["partition"]) == {"train", "validation", "test"}
print("Saved split checks passed.")


Saved split checks passed.


In [4]:
def file_sha256(path):
    digest = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


source_matches = (
    file_sha256(SOURCE) == metadata["source"]["sha256"]
)

manifest_matches = (
    file_sha256(MANIFEST)
    == metadata["artifacts_sha256"]["october_partition_manifest.csv"]
)

print("Source matches recorded version:", source_matches)
print("Manifest matches recorded version:", manifest_matches)

assert source_matches, "Source file differs from the recorded version."
assert manifest_matches, "Manifest differs from the recorded version."


Source matches recorded version: True
Manifest matches recorded version: True


In [5]:
categorical_cols = metadata["field_roles"]["categorical_predictors"]
continuous_cols = metadata["field_roles"]["continuous_predictors"]

feature_cols = categorical_cols + continuous_cols

forbidden_cols = {"id", "loss", "log1p_loss", "partition"}

assert categorical_cols == [f"cat{i}" for i in range(1, 117)]
assert continuous_cols == [f"cont{i}" for i in range(1, 15)]
assert len(set(feature_cols)) == 130
assert forbidden_cols.isdisjoint(feature_cols)

print("Categorical predictors:", len(categorical_cols))
print("Continuous predictors:", len(continuous_cols))
print("Total predictors:", len(feature_cols))
print("Forbidden columns included:", forbidden_cols.intersection(feature_cols))


Categorical predictors: 116
Continuous predictors: 14
Total predictors: 130
Forbidden columns included: set()


## 2 Training and validation inputs

Read only IDs and the 130 allowed predictors. The CSV reader passes through test rows, but only training and validation rows are retained. Loss values are never loaded. IDs stay in the row index for matching claims; they are not predictors.


In [6]:
development_manifest = manifest.loc[
    manifest["partition"].isin(["train", "validation"])
]

development_ids = set(development_manifest["id"])

column_types = {
    "id": "int64",
    **{col: "string" for col in categorical_cols},
    **{col: "float64" for col in continuous_cols},
}

development_chunks = []

for chunk in pd.read_csv(
    SOURCE,
    usecols=["id"] + feature_cols,
    dtype=column_types,
    chunksize=10_000,
):
    selected = chunk.loc[chunk["id"].isin(development_ids)].copy()
    development_chunks.append(selected)

development = pd.concat(
    development_chunks, ignore_index=True
).set_index("id")

del development_chunks, chunk, selected

print("Development predictor shape:", development.shape)
print("Unique claim IDs:", development.index.is_unique)
print("Missing predictor values:", development.isna().sum().sum())
print("Columns match allowed inputs:", set(development.columns) == set(feature_cols))


Development predictor shape: (169486, 130)
Unique claim IDs: True
Missing predictor values: 0
Columns match allowed inputs: True


In [7]:
train_ids = manifest.loc[manifest["partition"].eq("train"), "id"]

valid_ids = manifest.loc[manifest["partition"].eq("validation"), "id"]

X_train = development.loc[train_ids, feature_cols].copy()
X_valid = development.loc[valid_ids, feature_cols].copy()

assert X_train.shape == (141238, 130)
assert X_valid.shape == (28248, 130)

assert X_train.index.tolist() == train_ids.tolist()
assert X_valid.index.tolist() == valid_ids.tolist()

assert X_train.index.intersection(X_valid.index).empty

assert forbidden_cols.isdisjoint(X_train.columns)
assert forbidden_cols.isdisjoint(X_valid.columns)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)
print("Claim order matches manifest: True")
print("No shared claims between training and validation: True")
print("Forbidden columns excluded: True")

assert development.index.is_unique
assert set(development.index) == development_ids
del development


Training shape: (141238, 130)
Validation shape: (28248, 130)
Claim order matches manifest: True
No shared claims between training and validation: True
Forbidden columns excluded: True


## 3 Rare and unseen labels

Rare means fewer than 30 occurrences in training. Unseen means absent from training. Each field has its own rules. The `cat89` example shows how several rare labels become one group.

Validation counts use predictor labels only. A claim can have more than one affected field, so field occurrences and distinct claims are counted separately.


In [8]:
RARE_THRESHOLD = 30

training_category_counts = {
    col: X_train[col].value_counts()
    for col in categorical_cols
}

rare_levels = {
    col: counts.index[counts < RARE_THRESHOLD].tolist()
    for col, counts in training_category_counts.items()
}

rare_summary = pd.DataFrame({
    "training_levels": {
        col: len(counts)
        for col, counts in training_category_counts.items()
    },
    "rare_levels": {
        col: len(levels)
        for col, levels in rare_levels.items()
    },
})

print("Total rare levels:", rare_summary["rare_levels"].sum())
print("Fields with rare levels:", rare_summary["rare_levels"].gt(0).sum())

rare_summary.loc[rare_summary["rare_levels"].gt(0)].head(10)


Total rare levels: 352
Fields with rare levels: 30


,training_levels,rare_levels
cat15,2,1
cat22,2,1
cat70,2,1
cat73,3,1
cat74,3,1
cat88,4,1
cat89,8,5
cat90,7,3
cat91,8,1
cat92,7,3


In [9]:
example_col = "cat89"

example_counts = (
    training_category_counts[example_col]
    .rename_axis("original_label")
    .reset_index(name="training_count")
)

example_counts["is_rare"] = (
    example_counts["training_count"] < RARE_THRESHOLD
)

example_counts["grouped_label"] = example_counts["original_label"].where(
    ~example_counts["is_rare"],
    "__rare__",
)

example_counts


,original_label,training_count,is_rare,grouped_label
0,A,137816,False,A
1,B,3225,False,B
2,C,165,False,C
3,D,25,True,__rare__
4,E,3,True,__rare__
5,I,2,True,__rare__
6,H,1,True,__rare__
7,G,1,True,__rare__


In [10]:
rare_flags = pd.DataFrame({
    col: X_valid[col].isin(rare_levels[col])
    for col in categorical_cols
})

unseen_flags = pd.DataFrame({
    col: ~X_valid[col].isin(training_category_counts[col].index)
    for col in categorical_cols
})

print(
    "Validation claims with any training-rare label:",
    rare_flags.any(axis=1).sum(),
)

print(
    "Validation claims with any unseen label:",
    unseen_flags.any(axis=1).sum(),
)

print("Total unseen field occurrences:", unseen_flags.sum().sum())

unseen_by_field = unseen_flags.sum()
unseen_by_field.loc[unseen_by_field.gt(0)]


Validation claims with any training-rare label: 490
Validation claims with any unseen label: 12
Total unseen field occurrences: 13


cat105    1
cat107    1
cat109    1
cat110    1
cat114    1
cat116    8
dtype: int64

## 4 Input checks

Reject missing, renamed, duplicate, or extra columns, including target and ID columns. Continuous values must be numeric and finite. Missing categorical labels are allowed and handled by each approach.


In [11]:
def validate_feature_schema(X, expected_columns, continuous_columns):
    if not isinstance(X, pd.DataFrame):
        raise TypeError("Inputs must be a pandas DataFrame.")

    if not X.columns.is_unique:
        raise ValueError("Duplicate column names are not allowed.")

    missing = sorted(set(expected_columns) - set(X.columns))
    unexpected = sorted(set(X.columns) - set(expected_columns))

    if missing or unexpected:
        raise ValueError(
            f"Schema mismatch. Missing: {missing}; "
            f"unexpected: {unexpected}"
        )

    for col in continuous_columns:
        if not pd.api.types.is_numeric_dtype(X[col]):
            raise ValueError(f"{col} must contain numeric values.")

    continuous_values = X[continuous_columns].to_numpy(
        dtype="float64",
        na_value=np.nan,
    )

    if not np.isfinite(continuous_values).all():
        raise ValueError(
            "Continuous predictors contain missing or infinite values."
        )


In [12]:
class FeatureSchemaValidator(BaseEstimator, TransformerMixin):
    def __init__(self, expected_columns, continuous_columns):
        self.expected_columns = expected_columns
        self.continuous_columns = continuous_columns

    def fit(self, X, y=None):
        validate_feature_schema(
            X, self.expected_columns, self.continuous_columns
        )

        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        return self

    def transform(self, X):
        check_is_fitted(self, ["feature_names_in_"])

        validate_feature_schema(
            X, self.expected_columns, self.continuous_columns
        )

        return X.loc[:, self.expected_columns].copy()


In [13]:
validate_feature_schema(X_train, feature_cols, continuous_cols)
validate_feature_schema(X_valid, feature_cols, continuous_cols)

print("Training inputs pass schema checks: True")
print("Validation inputs pass schema checks: True")


Training inputs pass schema checks: True
Validation inputs pass schema checks: True


## 5 GLM preprocessing

The grouper learns training frequencies. The encoder keeps the learned categories but excludes the internal `__unseen__` placeholder. Unseen and missing labels use the rare group when one exists; otherwise, their field is all zeros.

Continuous values use training means and standard deviations. One-hot columns are stored sparsely to save memory. Each class is defined once below.


In [14]:
class RareCategoryGrouper(BaseEstimator, TransformerMixin):
    def __init__(self, min_count=30):
        self.min_count = min_count

    def fit(self, X, y=None):
        if X.eq("__rare__").any().any():
            raise ValueError("The reserved label '__rare__' already exists.")

        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        self.known_levels_ = {}
        self.rare_levels_ = {}

        for col in self.feature_names_in_:
            counts = X[col].value_counts(dropna=True)

            self.known_levels_[col] = counts.index.tolist()
            self.rare_levels_[col] = counts.index[
                counts < self.min_count
            ].tolist()

        return self

    def transform(self, X):
        check_is_fitted(self, ["known_levels_", "rare_levels_"])

        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError(
                "Categorical columns or their order differ from fitting."
            )

        transformed = X.astype("string").copy()

        for col in self.feature_names_in_:
            rare = self.rare_levels_[col]

            if rare:
                is_rare = transformed[col].isin(rare)
                is_unseen = ~transformed[col].isin(
                    self.known_levels_[col]
                )

                transformed[col] = transformed[col].mask(
                    is_rare | is_unseen,
                    "__rare__",
                )

            transformed[col] = transformed[col].fillna("__unseen__")

        return transformed


In [15]:
class PlaceholderSafeOneHotEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        self.active_columns_ = []
        categories = []

        for col in self.feature_names_in_:
            levels = sorted(
                label
                for label in X[col].unique()
                if label != "__unseen__"
            )

            if levels:
                self.active_columns_.append(col)
                categories.append(levels)

        self.encoder_ = None

        if self.active_columns_:
            self.encoder_ = OneHotEncoder(
                categories=categories,
                handle_unknown="ignore",
                drop=None,
                sparse_output=True,
                dtype="float64",
            )

            self.encoder_.fit(X[self.active_columns_])

        return self

    def transform(self, X):
        check_is_fitted(self, ["active_columns_", "encoder_"])

        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError(
                "Categorical columns or their order differ from fitting."
            )

        if self.encoder_ is None:
            return csr_matrix((len(X), 0), dtype="float64")

        return self.encoder_.transform(X[self.active_columns_])


In [16]:
categorical_pipeline = Pipeline([
    (
        "rare_grouping",
        RareCategoryGrouper(min_count=RARE_THRESHOLD),
    ),
    ("onehot", PlaceholderSafeOneHotEncoder()),
])

glm_preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", categorical_pipeline, categorical_cols),
        ("continuous", StandardScaler(), continuous_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

glm_preprocessing_pipeline = Pipeline([
    (
        "schema",
        FeatureSchemaValidator(
            expected_columns=feature_cols,
            continuous_columns=continuous_cols,
        ),
    ),
    ("preprocessing", glm_preprocessor),
])

X_train_checked = glm_preprocessing_pipeline.fit_transform(X_train)
X_valid_checked = glm_preprocessing_pipeline.transform(X_valid)

assert X_train_checked.shape == (141238, 808)
assert X_valid_checked.shape == (28248, 808)

assert np.isfinite(X_train_checked.data).all()
assert np.isfinite(X_valid_checked.data).all()


print("Training shape:", X_train_checked.shape)
print("Validation shape:", X_valid_checked.shape)
print("All processed values are finite: True")
print("GLM preprocessing checks passed.")


Training shape: (141238, 808)
Validation shape: (28248, 808)
All processed values are finite: True
GLM preprocessing checks passed.


In [17]:
fitted_scaler = (
    glm_preprocessing_pipeline.named_steps["preprocessing"]
    .named_transformers_["continuous"]
)

expected_means = X_train[continuous_cols].mean().to_numpy()

# ddof=0 calculates the population standard deviation, which is the convention StandardScaler uses. 
# Pandas defaults to ddof=1.
expected_scales = X_train[continuous_cols].std(ddof=0).to_numpy()

assert np.allclose(fitted_scaler.mean_, expected_means)
assert np.allclose(fitted_scaler.scale_, expected_scales)
assert fitted_scaler.n_samples_seen_ == len(X_train)

# .data contains the explicitly stored values in our sparse matrices. The other entries are zeros, which are already finite.
assert np.isfinite(X_train_checked.data).all()
assert np.isfinite(X_valid_checked.data).all()

print("Scaler means match training means: True")
print("Scaler scales match training standard deviations: True")
print("Rows used to fit scaler:", fitted_scaler.n_samples_seen_)
print("All processed training values are finite: True")
print("All processed validation values are finite: True")


Scaler means match training means: True
Scaler scales match training standard deviations: True
Rows used to fit scaler: 141238
All processed training values are finite: True
All processed validation values are finite: True


## 6 GLM fallback checks

Make one unseen label example per categorical field. Check the exact encoding: 30 fields use a rare column and 86 use all zeros. Missing label examples must behave the same way.

A small training example also checks that missing labels do not become a learned placeholder column.


In [18]:
unseen_examples = X_train.iloc[:len(categorical_cols)].copy()

for row_number, col in enumerate(categorical_cols):
    made_up_label = f"__test_unseen_{col}__"

    assert made_up_label not in training_category_counts[col].index

    column_number = unseen_examples.columns.get_loc(col)
    unseen_examples.iloc[row_number, column_number] = made_up_label

unseen_processed = glm_preprocessing_pipeline.transform(unseen_examples)

assert unseen_processed.shape == (116, 808)
assert np.isfinite(unseen_processed.data).all()

print("Categorical fields tested:", len(categorical_cols))
print("Processed shape:", unseen_processed.shape)
print("All test rows retained: True")
print("All processed values are finite: True")


Categorical fields tested: 116
Processed shape: (116, 808)
All test rows retained: True
All processed values are finite: True


In [19]:
fitted_categories = (
    glm_preprocessing_pipeline.named_steps["preprocessing"]
    .named_transformers_["categorical"]
)
fitted_grouper = fitted_categories.named_steps["rare_grouping"]
fitted_encoder = fitted_categories.named_steps["onehot"].encoder_

start = 0
rare_fallback_checks = 0
zero_fallback_checks = 0

for row_number, col in enumerate(categorical_cols):
    levels = fitted_encoder.categories_[row_number]
    stop = start + len(levels)

    actual = unseen_processed[
        row_number, start:stop
    ].toarray().ravel()

    expected = np.zeros(len(levels))

    if fitted_grouper.rare_levels_[col]:
        rare_position = list(levels).index("__rare__")
        expected[rare_position] = 1
        rare_fallback_checks += 1
    else:
        zero_fallback_checks += 1

    np.testing.assert_array_equal(
        actual,
        expected,
        err_msg=f"Unexpected unseen-label encoding for {col}",
    )

    start = stop

assert start == 794
assert rare_fallback_checks == 30
assert zero_fallback_checks == 86

print("Unseen labels mapped to rare group:", rare_fallback_checks)
print("Unseen labels encoded as all zeros:", zero_fallback_checks)
print("All fallback encodings verified: True")


Unseen labels mapped to rare group: 30
Unseen labels encoded as all zeros: 86
All fallback encodings verified: True


In [20]:
missing_examples = unseen_examples.copy()

for row_number, col in enumerate(categorical_cols):
    column_number = missing_examples.columns.get_loc(col)
    missing_examples.iloc[row_number, column_number] = pd.NA

missing_processed = glm_preprocessing_pipeline.transform(
    missing_examples
)

unseen_reference = glm_preprocessing_pipeline.transform(
    unseen_examples
)

assert missing_processed.shape == (116, 808)
assert np.isfinite(missing_processed.data).all()
assert (missing_processed != unseen_reference).nnz == 0

print("Categorical fields tested:", len(categorical_cols))
print("All test rows retained: True")
print("All processed values are finite: True")
print("Missing labels behave like unseen labels: True")


Categorical fields tested: 116
All test rows retained: True
All processed values are finite: True
Missing labels behave like unseen labels: True


In [21]:
training_with_missing = (
    X_train[categorical_cols].head(100).copy()
)

training_with_missing.loc[
    training_with_missing.index[0], "cat89"
] = pd.NA

probe_grouper = RareCategoryGrouper(min_count=RARE_THRESHOLD)
probe_grouped = probe_grouper.fit_transform(training_with_missing)

assert all(
    pd.notna(label)
    for label in probe_grouper.known_levels_["cat89"]
)

assert probe_grouped.shape == training_with_missing.shape
assert not probe_grouped.isna().any().any()

print("Fitting accepts missing categorical labels: True")
print("Missing values excluded from learned vocabulary: True")
print("Grouped output contains no missing values: True")


Fitting accepts missing categorical labels: True
Missing values excluded from learned vocabulary: True
Grouped output contains no missing values: True


In [22]:
example_training = pd.DataFrame({"example_cat": ["A", "A", "B", pd.NA]})
example_grouped = RareCategoryGrouper(min_count=1).fit_transform(example_training)
safe_example_encoder = PlaceholderSafeOneHotEncoder()
safe_example_encoded = safe_example_encoder.fit_transform(example_grouped)

assert safe_example_encoder.encoder_.categories_[0].tolist() == ["A", "B"]
assert safe_example_encoded.shape == (4, 2)
assert safe_example_encoded[-1].nnz == 0
print("Missing training labels do not create a placeholder column: True")


Missing training labels do not create a placeholder column: True


## 7 Invalid inputs and fitting boundaries

Deliberately try nine invalid inputs. Each must be rejected. Two small training subsets must learn different rare groups. A before and after fingerprint checks that validation does not change the fitted rules.

These subsets test pipeline behavior; they are not the team's cross validation fold setup.


In [23]:
sample = X_valid.head(3).copy()

invalid_examples = {
    "missing column": sample.drop(columns=["cont1"]),
    "renamed column": sample.rename(columns={"cat1": "renamed_cat1"}),
    "duplicate column": pd.concat([sample, sample[["cont1"]]], axis=1),
}

for col in ["id", "loss", "log1p_loss", "partition"]:
    invalid_examples[f"forbidden column: {col}"] = sample.assign(
        **{col: 0}
    )

for label, value in [
    ("missing continuous value", np.nan),
    ("infinite continuous value", np.inf),
]:
    example = sample.copy()
    example.loc[example.index[0], "cont1"] = value
    invalid_examples[label] = example

for label, example in invalid_examples.items():
    try:
        glm_preprocessing_pipeline.transform(example)
    except ValueError as error:
        print(f"PASS — {label}: {error}")
    else:
        raise AssertionError(f"Invalid input was accepted: {label}")


PASS — missing column: Schema mismatch. Missing: ['cont1']; unexpected: []
PASS — renamed column: Schema mismatch. Missing: ['cat1']; unexpected: ['renamed_cat1']
PASS — duplicate column: Duplicate column names are not allowed.
PASS — forbidden column: id: Schema mismatch. Missing: []; unexpected: ['id']
PASS — forbidden column: loss: Schema mismatch. Missing: []; unexpected: ['loss']
PASS — forbidden column: log1p_loss: Schema mismatch. Missing: []; unexpected: ['log1p_loss']
PASS — forbidden column: partition: Schema mismatch. Missing: []; unexpected: ['partition']
PASS — missing continuous value: Continuous predictors contain missing or infinite values.
PASS — infinite continuous value: Continuous predictors contain missing or infinite values.


In [24]:
fold_example = X_train.head(60).copy()
fold_example["cat89"] = ["A"] * 30 + ["B"] * 30

first_pipeline = clone(glm_preprocessing_pipeline)
second_pipeline = clone(glm_preprocessing_pipeline)

# First subset: 29 A labels and 30 B labels.
first_pipeline.fit(fold_example.iloc[1:])

# Second subset: 30 A labels and 29 B labels.
second_pipeline.fit(fold_example.iloc[:-1])

first_grouper = (
    first_pipeline.named_steps["preprocessing"]
    .named_transformers_["categorical"]
    .named_steps["rare_grouping"]
)

second_grouper = (
    second_pipeline.named_steps["preprocessing"]
    .named_transformers_["categorical"]
    .named_steps["rare_grouping"]
)

assert first_grouper.rare_levels_["cat89"] == ["A"]
assert second_grouper.rare_levels_["cat89"] == ["B"]

print("First subset's rare labels:", first_grouper.rare_levels_["cat89"])
print("Second subset's rare labels:", second_grouper.rare_levels_["cat89"])
print("Independent training-subset learning verified: True")


First subset's rare labels: ['A']
Second subset's rare labels: ['B']
Independent training-subset learning verified: True


In [25]:
state_before = hashlib.sha256(
    pickle.dumps(glm_preprocessing_pipeline)
).hexdigest()

validation_check = glm_preprocessing_pipeline.transform(X_valid)

state_after = hashlib.sha256(
    pickle.dumps(glm_preprocessing_pipeline)
).hexdigest()

assert state_before == state_after
assert validation_check.shape == (28248, 808)
assert np.isfinite(validation_check.data).all()
assert (validation_check != X_valid_checked).nnz == 0

print("Fitted pipeline unchanged after validation: True")
print("Validation output matches previous results: True")
print("All validation rows retained with finite values: True")


Fitted pipeline unchanged after validation: True
Validation output matches previous results: True
All validation rows retained with finite values: True


## 8 XGBoost preprocessing

Keep 130 columns. Known rare labels become `__rare__`; unseen and missing labels become missing categorical values. Each field uses an unordered vocabulary learned from training. Continuous values stay unchanged.


In [26]:
class NativeCategoricalPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, categorical_columns, min_count=30):
        self.categorical_columns = categorical_columns
        self.min_count = min_count

    def fit(self, X, y=None):
        if X[self.categorical_columns].eq("__rare__").any().any():
            raise ValueError("The reserved label '__rare__' already exists.")

        self.feature_names_in_ = X.columns.to_numpy(copy=True)
        self.n_features_in_ = X.shape[1]

        self.rare_levels_ = {}
        self.categories_ = {}

        for col in self.categorical_columns:
            counts = X[col].value_counts(dropna=True)

            rare = counts.index[counts < self.min_count].tolist()
            frequent = counts.index[counts >= self.min_count].tolist()

            self.rare_levels_[col] = rare
            self.categories_[col] = sorted(frequent)

            if rare:
                self.categories_[col].append("__rare__")

        return self

    def transform(self, X):
        check_is_fitted(self, ["rare_levels_", "categories_"])

        if list(X.columns) != list(self.feature_names_in_):
            raise ValueError(
                "Input columns or their order differ from fitting."
            )

        transformed = X.copy()

        for col in self.categorical_columns:
            labels = X[col].astype("string")

            grouped = labels.mask(
                labels.isin(self.rare_levels_[col]),
                "__rare__",
            )

            transformed[col] = pd.Categorical(
                grouped,
                categories=self.categories_[col],
                ordered=False,
            )

        return transformed


In [27]:
xgb_preprocessing_pipeline = Pipeline([
    (
        "schema",
        FeatureSchemaValidator(
            expected_columns=feature_cols,
            continuous_columns=continuous_cols,
        ),
    ),
    (
        "preprocessing",
        NativeCategoricalPreprocessor(
            categorical_columns=categorical_cols,
            min_count=RARE_THRESHOLD,
        ),
    ),
])

X_train_native = xgb_preprocessing_pipeline.fit_transform(X_train)
X_valid_native = xgb_preprocessing_pipeline.transform(X_valid)

print("Training shape:", X_train_native.shape)
print("Validation shape:", X_valid_native.shape)
print(
    "Categorical columns:",
    len(X_train_native.select_dtypes(include="category").columns),
)
print(
    "Continuous values unchanged:",
    X_train_native[continuous_cols].equals(X_train[continuous_cols])
    and X_valid_native[continuous_cols].equals(X_valid[continuous_cols]),
)


Training shape: (141238, 130)
Validation shape: (28248, 130)
Categorical columns: 116
Continuous values unchanged: True


## 9 XGBoost checks

Confirm matching vocabularies, rare grouping, claim order, and unchanged continuous values. The 13 unseen validation entries should affect 12 claims. Repeat the invalid input and fitting boundary checks for this branch.


In [28]:
native_transformer = (
    xgb_preprocessing_pipeline.named_steps["preprocessing"]
)

for col in categorical_cols:
    expected_categories = native_transformer.categories_[col]

    assert X_train_native[col].cat.categories.tolist() == expected_categories
    assert X_valid_native[col].cat.categories.tolist() == expected_categories

    assert native_transformer.rare_levels_[col] == rare_levels[col]

    np.testing.assert_array_equal(
        X_valid_native[col].isna().to_numpy(),
        unseen_flags[col].to_numpy(),
    )

assert X_train_native.index.equals(X_train.index)
assert X_valid_native.index.equals(X_valid.index)

print("Training and validation vocabularies match: True")
print("Rare-label rules match training counts: True")
print("Missing validation entries:", X_valid_native[categorical_cols].isna().sum().sum())
print(
    "Validation claims with missing categories:",
    X_valid_native[categorical_cols].isna().any(axis=1).sum(),
)
print("Claim IDs and row order preserved: True")


Training and validation vocabularies match: True
Rare-label rules match training counts: True
Missing validation entries: 13
Validation claims with missing categories: 12
Claim IDs and row order preserved: True


In [29]:
for col in categorical_cols:
    train_rare_mask = X_train[col].isin(rare_levels[col])
    valid_rare_mask = X_valid[col].isin(rare_levels[col])

    assert (
        X_train_native.loc[train_rare_mask, col] == "__rare__"
    ).all()

    assert (
        X_valid_native.loc[valid_rare_mask, col] == "__rare__"
    ).all()

native_unseen = xgb_preprocessing_pipeline.transform(unseen_examples)
native_missing = xgb_preprocessing_pipeline.transform(missing_examples)

for row_number, col in enumerate(categorical_cols):
    assert pd.isna(native_unseen.iloc[row_number][col])
    assert pd.isna(native_missing.iloc[row_number][col])

assert native_unseen.index.equals(unseen_examples.index)
assert native_missing.index.equals(missing_examples.index)

assert native_unseen.shape == (116, 130)
assert native_missing.shape == (116, 130)

assert native_unseen[continuous_cols].equals(
    unseen_examples[continuous_cols]
)

assert native_missing[continuous_cols].equals(
    missing_examples[continuous_cols]
)

print("Known rare labels map to __rare__: True")
print("Unseen labels become missing in all 116 fields: True")
print("Missing labels remain missing in all 116 fields: True")
print("All test rows and claim IDs retained: True")
print("Continuous values unchanged: True")


Known rare labels map to __rare__: True
Unseen labels become missing in all 116 fields: True
Missing labels remain missing in all 116 fields: True
All test rows and claim IDs retained: True
Continuous values unchanged: True


In [30]:
# Check invalid-input rejection.
rejected_count = 0

for label, example in invalid_examples.items():
    try:
        xgb_preprocessing_pipeline.transform(example)
    except ValueError:
        rejected_count += 1
    else:
        raise AssertionError(f"Invalid input was accepted: {label}")

assert rejected_count == 9

# Check that validation does not update the fitted pipeline.
state_before = hashlib.sha256(
    pickle.dumps(xgb_preprocessing_pipeline)
).hexdigest()

validation_native_check = xgb_preprocessing_pipeline.transform(X_valid)

state_after = hashlib.sha256(
    pickle.dumps(xgb_preprocessing_pipeline)
).hexdigest()

assert state_before == state_after
pd.testing.assert_frame_equal(validation_native_check, X_valid_native)

# Reuse our artificial training-subset example.
first_native_pipeline = clone(xgb_preprocessing_pipeline)
second_native_pipeline = clone(xgb_preprocessing_pipeline)

first_native_pipeline.fit(fold_example.iloc[1:])
second_native_pipeline.fit(fold_example.iloc[:-1])

first_native = first_native_pipeline.named_steps["preprocessing"]
second_native = second_native_pipeline.named_steps["preprocessing"]

assert first_native.rare_levels_["cat89"] == ["A"]
assert second_native.rare_levels_["cat89"] == ["B"]

print("Invalid inputs correctly rejected:", rejected_count)
print("Fitted pipeline unchanged after validation: True")
print("Validation output unchanged: True")
print("Independent training-subset learning verified: True")


Invalid inputs correctly rejected: 9
Fitted pipeline unchanged after validation: True
Validation output unchanged: True
Independent training-subset learning verified: True


## 10 Prediction compatibility

Attach small models to fresh preprocessing copies and fit on 1,000 training rows with artificial targets. Check one finite prediction per row for validation, unseen label, and missing label examples.

These predictions do not measure accuracy. No validation loss, MAE, target transformation, or tuning decision is used. XGBoost uses native categorical support with `hist`.


In [31]:
compatibility_X = X_train.head(1000).copy()
compatibility_y = np.arange(len(compatibility_X), dtype=float) % 17

xgb_compatibility_pipeline = Pipeline([
    ("preprocessing", clone(xgb_preprocessing_pipeline)),
    (
        "model",
        xgb.XGBRegressor(
            n_estimators=5,
            max_depth=2,
            tree_method="hist",
            enable_categorical=True,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=2,
        ),
    ),
])

xgb_compatibility_pipeline.fit(compatibility_X, compatibility_y)

prediction_examples = {
    "validation": X_valid,
    "unseen labels": unseen_examples,
    "missing labels": missing_examples,
}

for label, examples in prediction_examples.items():
    predictions = xgb_compatibility_pipeline.predict(examples)

    assert predictions.shape == (len(examples),)
    assert np.isfinite(predictions).all()

    print(f"PASS — {label}: {len(predictions)} finite predictions")


PASS — validation: 28248 finite predictions
PASS — unseen labels: 116 finite predictions
PASS — missing labels: 116 finite predictions


In [32]:
glm_compatibility_pipeline = Pipeline([
    ("preprocessing", clone(glm_preprocessing_pipeline)),
    (
        "model",
        Ridge(
            alpha=1.0,
            solver="lsqr",
        ),
    ),
])

glm_compatibility_pipeline.fit(
    compatibility_X,
    compatibility_y,
)

for label, examples in prediction_examples.items():
    predictions = glm_compatibility_pipeline.predict(examples)

    assert predictions.shape == (len(examples),)
    assert np.isfinite(predictions).all()

    print(f"PASS — {label}: {len(predictions)} finite predictions")


PASS — validation: 28248 finite predictions
PASS — unseen labels: 116 finite predictions
PASS — missing labels: 116 finite predictions


## 11 Claim alignment

Shuffle 200 validation claims, predict again, and match results back by ID. Each claim must receive the same prediction regardless of row order.


In [33]:
alignment_examples = X_valid.head(200).copy()
shuffled_examples = alignment_examples.sample(
    frac=1,
    random_state=42,
)

for label, pipeline in [
    ("GLM", glm_compatibility_pipeline),
    ("XGBoost", xgb_compatibility_pipeline),
]:
    original_predictions = pd.Series(
        pipeline.predict(alignment_examples),
        index=alignment_examples.index,
    )

    shuffled_predictions = pd.Series(
        pipeline.predict(shuffled_examples),
        index=shuffled_examples.index,
    )

    restored_predictions = shuffled_predictions.reindex(
        original_predictions.index
    )

    assert original_predictions.index.is_unique

    np.testing.assert_allclose(
        restored_predictions.to_numpy(),
        original_predictions.to_numpy(),
        rtol=1e-7,
        atol=1e-10,
    )

    print(f"PASS — {label}: predictions stay aligned to claim IDs")


PASS — GLM: predictions stay aligned to claim IDs
PASS — XGBoost: predictions stay aligned to claim IDs


## 12 Handoff summary

The final summary records the split, counts, output sizes, and current environment. Rare and unseen groups overlap: 490 + 12 − 3 = 499 distinct affected validation claims.

The two fitted preprocessing objects are `glm_preprocessing_pipeline` and `xgb_preprocessing_pipeline`. Teammates should clone them before fitting their own training folds. Actual model training, evaluation, independent review, and Gate 2 approval remain separate work.


In [34]:
has_rare = rare_flags.any(axis=1)
has_unseen = unseen_flags.any(axis=1)

handoff_summary = {
    "split_version": metadata["split"]["version"],
    "rare_threshold": RARE_THRESHOLD,
    "training_claims": len(X_train),
    "validation_claims": len(X_valid),
    "validation_claims_with_training_rare_labels": int(has_rare.sum()),
    "validation_claims_with_unseen_labels": int(has_unseen.sum()),
    "validation_unseen_field_occurrences": int(
        unseen_flags.sum().sum()
    ),
    "validation_claims_with_both": int(
        (has_rare & has_unseen).sum()
    ),
    "validation_claims_with_either": int(
        (has_rare | has_unseen).sum()
    ),
    "glm_output_columns": X_train_checked.shape[1],
    "xgboost_output_columns": X_train_native.shape[1],
    "python_version": sys.version.split()[0],
    "package_versions": {
        package: version(package)
        for package in ["pandas", "numpy", "scipy", "scikit-learn", "xgboost"]
    },
    "prediction_checks": "Artificial targets; no validation loss used",
}

print(json.dumps(handoff_summary, indent=2))


{
  "split_version": "october-split-v1",
  "rare_threshold": 30,
  "training_claims": 141238,
  "validation_claims": 28248,
  "validation_claims_with_training_rare_labels": 490,
  "validation_claims_with_unseen_labels": 12,
  "validation_unseen_field_occurrences": 13,
  "validation_claims_with_both": 3,
  "validation_claims_with_either": 499,
  "glm_output_columns": 808,
  "xgboost_output_columns": 130,
  "python_version": "3.13.2",
  "package_versions": {
    "pandas": "2.3.2",
    "numpy": "2.3.3",
    "scipy": "1.18.1",
    "scikit-learn": "1.7.2",
    "xgboost": "3.1.1"
  },
  "prediction_checks": "Artificial targets; no validation loss used"
}
